# Model Comparison

Aggregates the test-set metrics produced by the three model notebooks into a
single side-by-side table, and verifies that all three models were evaluated on
exactly the same test rows (identical scenario-grouped split).

Per-model predictions vs actuals are saved by each model notebook as
`<model>_predictions.csv` in `output/`.

## 1. Load Model Metrics

In [1]:
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("../output")

MODELS = {
    "Logistic Regression": "logistic_regression",
    "Random Forest": "random_forest",
    "XGBoost": "xgboost",
}

metric_frames = {
    name: pd.read_csv(OUTPUT_DIR / f"{slug}_metrics.csv").set_index("Metric")["Score"]
    for name, slug in MODELS.items()
}

comparison_table = pd.DataFrame(metric_frames).T.rename_axis("Model").round(4)

print("Test-set metrics (identical scenario-grouped train/test split for all models):")
print(comparison_table.to_string())

Test-set metrics (identical scenario-grouped train/test split for all models):
Metric               Accuracy  Balanced Accuracy  Precision (macro)  Recall (macro)  F1 (macro)  Recall (Critical)  ROC-AUC (macro OVR)
Model                                                                                                                                  
Logistic Regression    0.8608             0.8610             0.8513          0.8610      0.8554             0.8767               0.9636
Random Forest          0.8382             0.8215             0.8238          0.8215      0.8226             0.7935               0.9566
XGBoost                0.8680             0.8624             0.8466          0.8624      0.8537             0.8692               0.9731


## 2. Best Model per Metric

In [2]:
best_per_metric = pd.DataFrame(
    {
        "Best Model": comparison_table.idxmax(),
        "Best Score": comparison_table.max().round(4),
    }
).rename_axis("Metric")

print(best_per_metric.to_string())

                              Best Model  Best Score
Metric                                              
Accuracy                         XGBoost      0.8680
Balanced Accuracy                XGBoost      0.8624
Precision (macro)    Logistic Regression      0.8513
Recall (macro)                   XGBoost      0.8624
F1 (macro)           Logistic Regression      0.8554
Recall (Critical)    Logistic Regression      0.8767
ROC-AUC (macro OVR)              XGBoost      0.9731


## 3. Split Consistency Check

All three prediction files must cover exactly the same bank-scenario rows in
the same order, confirming the comparison uses one shared test set.

In [3]:
prediction_keys = {
    name: pd.read_csv(OUTPUT_DIR / f"{slug}_predictions.csv")[["bank_id", "scenario_id"]]
    for name, slug in MODELS.items()
}

reference = next(iter(prediction_keys.values()))
consistent = all(keys.equals(reference) for keys in prediction_keys.values())

print("All three models evaluated on the same test rows:", consistent)

All three models evaluated on the same test rows: True


## 4. Save Comparison Table

In [4]:
comparison_path = OUTPUT_DIR / "model_comparison.csv"
comparison_table.to_csv(comparison_path)

print("Saved:", comparison_path)

Saved: ..\output\model_comparison.csv
